# Phase 1 — EDA: Durian Disease Dataset

Senior project: 10-class durian disease classifier. This notebook verifies the
**real** dataset (the proposal *claims* 4,000 images / 400 per class — we trust
nothing and count it ourselves), and records the Phase-1 **Decision Logs**.

All heavy logic lives in `src/eda.py` (no magic numbers; everything from
`config.yaml`) so the notebook and a headless run are identical.


In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')   # run from repo root
sys.path.insert(0, os.getcwd())
from src.utils import load_config
from src import eda, data
cfg = load_config('config.yaml')
cfg['data']['image_format'], cfg['paths']['data_root']


## 1.1 Verify class counts (the TRUTH)

The proposal says ~400/class, balanced. The dataset paper reports ~5,452 images
with ~405–427/class (mildly imbalanced). We report what is actually on disk.


In [ ]:
df = data.discover_images(cfg)
summary_counts = eda.class_counts(df, cfg['data']['classes'], __import__('pathlib').Path(cfg['paths']['eda_dir']))
import pandas as pd; pd.Series(summary_counts).to_frame('count')


> **Read the bar chart in `outputs/eda/class_counts.png`.** If max/min ratio is
> close to 1.0 the set is balanced (use macro-F1 anyway for per-class fairness);
> if it exceeds ~1.1, note the mild imbalance in the report and revisit class
> weighting in Phase 11.


## 1.2 Resolution & aspect ratio  →  resize strategy [DL-RESIZE]


In [ ]:
res = eda.resolution_stats(df, __import__('pathlib').Path(cfg['paths']['eda_dir']))
res


**Decision Log [DL-RESIZE]** — *Decision:* resize/crop to **224×224, bilinear**.
*Why:* all three backbones were ImageNet-pretrained at 224, so 224 reuses their
learned spatial priors directly; field photos here are far larger than 224 (see
median above) so downscaling, not upscaling, dominates → no fabricated detail.
*Alternatives:* (a) 384/512 — rejected: ~3–5× compute for marginal gain on only
~5k images and longer epochs on a single GPU; (b) bicubic — minor sharpening but
can ring on lesion edges; bilinear is the timm default the weights expect;
(c) keep native aspect (pad) — rejected: wastes resolution and the backbones
expect square inputs.


## 1.3 Brightness / colour statistics  →  normalization + safe augmentations


In [ ]:
bn = eda.brightness_and_norm(df, cfg['data']['classes'], __import__('pathlib').Path(cfg['paths']['eda_dir']))
bn['dataset_mean'], bn['dataset_std']


**Decision Log [DL-NORM]** — *Decision:* use **ImageNet mean/std** for training.
*Why:* we fine-tune ImageNet-pretrained backbones; matching the pretraining input
distribution keeps the early conv/attention filters in their calibrated range and
stabilises transfer on a small set. *Alternative:* dataset-computed stats (printed
above) — rejected for transfer because re-centering shifts inputs away from what
the pretrained filters expect, hurting early epochs for negligible benefit on
natural RGB photos. We keep the dataset stats on record for an ablation only.

**Why colour stats matter here:** classes **Yellow leaf, Pink disease, Sooty mold**
are *defined by colour*. The per-class channel means (in `bn`) confirm colour is
class-discriminative → this is the evidence behind turning **hue/saturation jitter
OFF** in Phase 3 ([DL-AUG-OFF]).


## 1.4 Duplicate / near-duplicate detection (leakage risk) [DL-GROUP]


In [ ]:
dups = eda.duplicate_report(cfg, df, __import__('pathlib').Path(cfg['paths']['eda_dir']))
dups


**Decision Log [DL-GROUP]** — *Decision:* detect near-duplicates with perceptual
hashing (pHash, Hamming ≤ `near_dup_hamming`) and treat each near-duplicate cluster
as one **group**; Phase 2 keeps a group entirely within one split. *Why:* a
single-orchard dataset contains many near-identical shots of the same fruit/scene;
if those straddle train/test the model can match the *photo*, not the *disease*,
and test accuracy is inflated. *Cross-class* near-duplicate groups (reported above)
are a direct artifact/leakage signal feeding the project's central question.
*Alternatives:* exact-hash dedup (misses re-encodes/crops); embedding-based dedup
(heavier, needs a model) — pHash is the simplest defensible choice at this scale.


## 1.5 Background / artifact audit  →  the project's central question

Sample a few images per class and look: do backgrounds / capture setup correlate
with the label? This sets up Phase 8 (Grad-CAM) and Phase 9 (OOD).


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
classes = cfg['data']['classes']
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3*len(classes)))
for r, c in enumerate(classes):
    paths = df[df.label==c]['path'].head(4).tolist()
    for k in range(4):
        ax = axes[r, k]; ax.axis('off')
        if k < len(paths):
            ax.imshow(Image.open(paths[k]).convert('RGB'))
            if k==0: ax.set_title(c, loc='left', fontsize=9)
plt.tight_layout(); plt.show()


**Audit verdict (fill after looking):** note any class where the *background*
(orchard floor, hand, ruler, lab bench) is visually consistent. Likely-confusable
pairs to watch in the confusion matrix: **Sooty mold ↔ Mealybug** (sooty mold
follows mealybug), **Fruit rot ↔ Stem cracking gummosis** (both *Phytophthora*),
**Stem blight ↔ Canker** (both bark/branch damage).


## 1.6 JPEG vs PNG — primary training input [DL-FORMAT]

**Decision:** train primarily on the **raw JPEG** (full frame), use the **cropped
PNG** as a background-reliance *probe*, not the main input.
**Why:** the project's stated goal is robustness to real-world messiness and
answering *“disease features vs dataset artifacts?”*. Raw JPEGs keep the noisy
backgrounds that (a) match deployment (farmer/web photos have backgrounds) and
(b) make the central question testable — if the model shortcuts on background,
Grad-CAM (Phase 8) and the ID→OOD gap (Phase 9) will expose it.
**Alternatives:** train on cropped PNG — rejected as *primary* because removing
background hides the very shortcut we want to measure and widens the gap to
background-rich OOD images; using both merged — rejected as it confounds the
format effect. **Use of the other format:** a clean ablation — train-on-JPEG /
test-on-PNG vs the reverse — directly quantifies background reliance (switch via
`config.data.image_format`).


## Phase-1 summary

- Real counts, resolution, colour stats, and duplicates are written to
  `outputs/eda/` (`eda_summary.json` is the machine-readable digest).
- Decision Logs recorded: [DL-FORMAT], [DL-RESIZE], [DL-NORM], [DL-GROUP].
- Next: **Phase 2** builds the group-aware stratified split
  (`python -m src.train` creates `outputs/splits.json` on first run).
